In [ ]:
%pip install xgboost lightgbm statsmodels

In [ ]:
%run ../globalvariables

In [ ]:
%run ../core/base_model

In [ ]:
%run ../core/models

In [ ]:
%run ../core/evaluation

In [ ]:
%run ../core/tuning

In [ ]:
%run ../config/logger_config

In [ ]:
import yaml

In [ ]:
# Target gas and district widgets
dbutils.widgets.dropdown("magnitud_target", "no2", [
    "no2", "no", "nox", "pm10", "pm2_5", "o3", "so2", "co",
    "tol", "ben", "ebe", "ch4", "nmhc", "tch",
])
dbutils.widgets.dropdown("distrito", "16", [str(i) for i in range(1, 22)])
MAGNITUD_TARGET = dbutils.widgets.get("magnitud_target")
DISTRITO = dbutils.widgets.get("distrito")
DISTRITO_TAG = f"D{DISTRITO.zfill(2)}"

MAGNITUD_MAP = {
    "no2": "NO2", "no": "NO", "nox": "NOx", "pm10": "PM10", "pm2_5": "PM2.5",
    "o3": "O3", "so2": "SO2", "co": "CO", "tol": "TOL", "ben": "BEN",
    "ebe": "EBE", "ch4": "CH4", "nmhc": "NMHC", "tch": "TCH",
}
MAGNITUD_VALUE = MAGNITUD_MAP[MAGNITUD_TARGET]
DOMAIN = f"{MAGNITUD_TARGET}_{DISTRITO_TAG}"
NOTEBOOK = "ml/domains/forecast_gas_distrito"
errors = []

In [ ]:
# Traffic columns pivoted wide
TRAFFIC_COLS = [
    f"{franja}_{metric}"
    for franja in ("Manana", "Tarde", "Noche")
    for metric in ("intensidad_media", "ocupacion_media", "carga_pico", "vmed_media")
]
CALENDAR_COLS = ["dia_semana", "mes", "es_finde", "dow_sin", "dow_cos", "mes_sin", "mes_cos"]
LAG_DAYS = (1, 2, 3, 7, 14)
LAG_COLS = [f"lag_{n}" for n in LAG_DAYS]
ROLLING_COLS = ["rolling_mean_7d", "rolling_mean_30d", "rolling_std_7d"]
STATIC_COLS = ["n_estaciones", "es_festivo", "num_fabricas_distrito"]
TARGET_COL = "valor_medio"
FEATURE_COLS = STATIC_COLS + LAG_COLS + ROLLING_COLS + CALENDAR_COLS + TRAFFIC_COLS

In [ ]:
def calendar_feats(fechas):
    # Known in advance, safe to build for future dates
    dow, mes = fechas.dt.dayofweek, fechas.dt.month
    return pd.DataFrame({
        "dia_semana": dow.astype(float),
        "mes": mes.astype(float),
        "es_finde": (dow >= 5).astype(float),
        "dow_sin": np.sin(2 * np.pi * dow / 7),
        "dow_cos": np.cos(2 * np.pi * dow / 7),
        "mes_sin": np.sin(2 * np.pi * mes / 12),
        "mes_cos": np.cos(2 * np.pi * mes / 12),
    }, index=fechas.index)


# Load features and lags, single district table
FEATURE_TABLE = f"features_{MAGNITUD_TARGET}_{DISTRITO_TAG}_{FORECAST_HORIZON_GAS_DAYS}"
gas = spark.table(f"{ML_TABLE}.{FEATURE_TABLE}")
gas = add_lag_features(gas, TARGET_COL, partition_cols=("cod_dis",), lags=LAG_DAYS).toPandas()

# toPandas gives datetime.date and no ordering guarantee
gas["fecha"] = pd.to_datetime(gas["fecha"])
gas["es_festivo"] = gas["es_festivo"].astype(float)
gas = gas.sort_values("fecha").reset_index(drop=True)
gas = gas.join(calendar_feats(gas["fecha"]))

train_pdf, test_pdf = temporal_split(gas)

# Constant columns carry no signal and make SARIMAX singular
FEATURE_COLS = [c for c in FEATURE_COLS if train_pdf[c].nunique(dropna=False) > 1]
X_train, y_train = train_pdf[FEATURE_COLS], train_pdf[TARGET_COL]
X_test, y_test = test_pdf[FEATURE_COLS], test_pdf[TARGET_COL]
logger.info(f"{DOMAIN}: {len(train_pdf)} train rows, {len(test_pdf)} test rows, {len(FEATURE_COLS)} features")

In [ ]:
from datetime import date

# One experiment folder per gas, district and date
mlflow.set_registry_uri("databricks-uc")
EXPERIMENT_PATH = f"{ML_EXPERIMENT_BASE}/{DOMAIN}_{date.today().isoformat()}"
mlflow.set_experiment(EXPERIMENT_PATH)
logger.info(f"experiment: {EXPERIMENT_PATH}")

# Hyperparameter ranges per model
with open("../config/model_search_space_gas_distrito.yml") as f:
    search_config = yaml.safe_load(f)

In [ ]:
# Holidays known in advance
calendario = (
    spark.table(f"{GOLD_TABLE}.dim_fecha")
    .select("fecha", "es_festivo")
    .toPandas()
    .set_index("fecha")["es_festivo"]
)
calendario.index = pd.to_datetime(calendario.index)

history = gas.set_index(["cod_dis", "fecha"])[TARGET_COL]

# Last known row for this district
seeds = gas.groupby("cod_dis").tail(1).set_index("cod_dis")

# Skip if this district's data has gone stale, same guard as forecast_gas.ipynb
STALE_CUTOFF_DAYS = 14
global_max_fecha = gas["fecha"].max()
stale_mask = seeds["fecha"] < (global_max_fecha - pd.Timedelta(days=STALE_CUTOFF_DAYS))
if stale_mask.any():
    logger.warning(f"distrito {DISTRITO} has stale data, no forecast will be produced")
seeds = seeds.loc[~stale_mask]

In [ ]:
# Same month, same weekday, train only to keep the backtest honest
SIMILAR_COLS = [c for c in ROLLING_COLS + TRAFFIC_COLS if c in FEATURE_COLS]
similar_avg = (
    train_pdf.assign(mes=train_pdf["fecha"].dt.month, dia=train_pdf["fecha"].dt.dayofweek)
    .groupby(["cod_dis", "mes", "dia"])[SIMILAR_COLS]
    .mean()
)
logger.info(f"similar day table: {len(similar_avg)} combos")

In [ ]:
def feature_row(cod_dis, seed, target_fecha, known):
    # Seed as fallback when that month/weekday combo is unseen
    key = (cod_dis, target_fecha.month, target_fecha.dayofweek)
    similar = similar_avg.loc[key] if key in similar_avg.index else seed

    row = {c: similar[c] for c in SIMILAR_COLS}
    row["n_estaciones"] = seed["n_estaciones"]
    row["num_fabricas_distrito"] = seed["num_fabricas_distrito"]
    row["es_festivo"] = float(calendario.get(target_fecha, False))
    row.update(calendar_feats(pd.Series([target_fecha])).iloc[0].to_dict())

    # Chained predictions win over history, 0.0 is a valid value so test for None
    for n in LAG_DAYS:
        lag_fecha = target_fecha - pd.Timedelta(days=n)
        chained = known.get((cod_dis, lag_fecha))
        row[f"lag_{n}"] = history.get((cod_dis, lag_fecha)) if chained is None else chained

    # Force numeric dtype: a missing lag (None) makes a 1-row
    # DataFrame infer dtype=object, which some models reject
    return pd.DataFrame([row])[FEATURE_COLS].astype(float)


def recursive_forecast(fit_model, origins, horizon=FORECAST_HORIZON_GAS_DAYS):
    fit_model.reset_forecast()
    known = {}
    pred_rows = []

    for cod_dis, seed in origins.iterrows():
        for h in range(1, horizon + 1):
            target_fecha = seed["fecha"] + pd.Timedelta(days=h)
            X_pred = feature_row(cod_dis, seed, target_fecha, known)
            valor_predicho = float(fit_model.predict(X_pred)[0])
            known[(cod_dis, target_fecha)] = valor_predicho

            pred_rows.append({
                "cod_dis": cod_dis,
                "fecha": target_fecha,
                "valor_predicho": valor_predicho,
                "_h": h,
            })

    return pd.DataFrame(pred_rows)

In [ ]:
def backtest_recursive(fit_model, horizon=FORECAST_HORIZON_GAS_DAYS):
    # Rolling origin over the test window, same chained forecast we ship.
    # The 1-step test MAE feeds each row real lags, so it flatters any model
    # that leans on lag_1, this is what the 7 day product actually costs.
    origin = train_pdf.groupby("cod_dis").tail(1).set_index("cod_dis")
    blocks = [test_pdf.iloc[i:i + horizon] for i in range(0, len(test_pdf), horizon)]
    errs = {h: [] for h in range(1, horizon + 1)}

    for block in blocks:
        if len(block) < horizon:
            break
        pred = recursive_forecast(fit_model, origin, horizon)
        merged = pred.merge(block[["cod_dis", "fecha", TARGET_COL]], on=["cod_dis", "fecha"])
        for h, group in merged.groupby("_h"):
            errs[h].extend((group["valor_predicho"] - group[TARGET_COL]).abs())
        # Move the fitted state forward on real observations, no refit
        fit_model.advance(block[FEATURE_COLS], block[TARGET_COL])
        origin = block.groupby("cod_dis").tail(1).set_index("cod_dis")

    flat = [e for v in errs.values() for e in v]
    if not flat:
        raise RuntimeError("backtest produced no comparable points")

    metrics = {f"mae_h{h}": float(np.mean(v)) for h, v in errs.items() if v}
    metrics[f"mae_{horizon}d"] = float(np.mean(flat))
    return metrics

In [ ]:
HORIZON_METRIC = f"mae_{FORECAST_HORIZON_GAS_DAYS}d"

# One flat run per model
best_per_model = []

for model_type, cfg in search_config["models"].items():
    try:
        model_class = MODEL_REGISTRY[model_type]
        param_grid = model_class.get_search_space(cfg.get("search_space"), cfg.get("n_grid_points"))
        run_id, mae, rmse, smape, params = run_gridsearch(
            model_class, model_type, param_grid,
            X_train, y_train, X_test, y_test, log_model=False,
            run_name=f"{DOMAIN}_{model_type}",
            n_iter=cfg.get("n_iter"), cv_splits=cfg.get("cv_splits", 5),
        )

        # Score the chained forecast, not just the 1 step fit
        candidate = model_class.build(params)
        candidate.fit(X_train, y_train)
        horizon_metrics = backtest_recursive(candidate)
        with mlflow.start_run(run_id=run_id):
            for k, v in horizon_metrics.items():
                mlflow.log_metric(k, v)

        best_per_model.append((model_type, run_id, horizon_metrics[HORIZON_METRIC], mae, params))
        logger.info(
            f"{model_type}: {HORIZON_METRIC}={horizon_metrics[HORIZON_METRIC]:.3f} "
            f"mae_1step={mae:.3f} rmse={rmse:.3f} smape={smape:.3f} params={params}"
        )
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        logger.error(f"fail {model_type}: {type(e).__name__}: {e}")

if not best_per_model:
    raise RuntimeError(f"{DOMAIN}: every model failed, see infra.error_logs")

In [ ]:
# Winner by chained 7 day error, refit on train plus test
best_model_type, best_run_id, best_horizon_mae, best_mae, best_params = min(
    best_per_model, key=lambda t: t[2]
)
model = MODEL_REGISTRY[best_model_type].build(best_params)
model.fit(gas[FEATURE_COLS], gas[TARGET_COL])
logger.info(f"winner: {best_model_type} {HORIZON_METRIC}={best_horizon_mae:.3f} mae_1step={best_mae:.3f}")

pred_df = recursive_forecast(model, seeds)

In [ ]:
# Reopen the winning run to log the model and mark it as best
with mlflow.start_run(run_id=best_run_id):
    mlflow.sklearn.log_model(
        model.estimator, artifact_path="model",
        input_example=X_train.head(3),
        signature=mlflow.models.infer_signature(X_train, y_train),
    )
    MlflowClient().set_tag(best_run_id, "mlflow.runName", f"{DOMAIN}_{best_model_type}_best")

# Promote only if better, one registered model per gas and district
version, promoted = promote_if_better(DOMAIN, best_run_id, best_horizon_mae, metric_key=HORIZON_METRIC)
logger.info(f"registered v{version}, promoted={promoted}")

In [ ]:
# One table per gas, district and horizon
FORECAST_TABLE = f"forecast_{MAGNITUD_TARGET}_{DISTRITO_TAG}_{FORECAST_HORIZON_GAS_DAYS}"
out_df = pred_df.drop(columns="_h").rename(columns={"cod_dis": "distrito"})
out_df["fecha"] = out_df["fecha"].dt.date
write_ml(spark.createDataFrame(out_df), FORECAST_TABLE, mode="overwrite")

log_errors(errors)
logger.info(f"{FORECAST_TABLE} updated: {len(out_df)} rows, run={best_run_id}")